# Project Impact Lense — Sentiment Analysis Pipeline (cleaned)

Cleaned version of the group notebook for portfolio purposes. Removed: duplicate plotting cells, an abandoned first aggregation method superseded by the `literal_eval` approach below, an unused heatmap attempt, and inline team-chat comments.

Full methodology and findings: see repo `README.md`.

**Pipeline:** BigQuery extraction → text filtering (length, peak months) → NLTK cleaning (tokenisation, lemmatisation) → Hugging Face emotion classification (`SamLowe/roberta-base-go_emotions`) → per-party aggregation → visualisation of 4 retained emotions (love, caring, disgust, fear).


# Installation and Loading of the Dataframe

In [ ]:
import os
os.getcwd()

In [ ]:
os.listdir(path=os.getcwd())

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pip install transformers

# **Big Query Import**

In [ ]:
df=pd.read_gbq("SELECT * FROM `transform.sentiment_analysis`", project_id="project-impact-lense")

df

# Data Transformation

In [ ]:
df.info()

In [ ]:
#change date column format

df['ad_delivery_start_time'] = pd.to_datetime(df['ad_delivery_start_time'])

In [ ]:
df.info()

In [ ]:
# rename the translated column

df = df.rename(columns={'EN_translations_ad_creative_bodies':'translated_posts'})
df.head

# **Filter on number of characters**

In [ ]:
# Let's define the mimimum number of characters in ads
minimum_size_characters = 20

# How many ads have less than this minimum number of characters?
f"Number of ads with less than {minimum_size_characters} characters: {(df['ad_creative_bodies'].apply(len) < minimum_size_characters).sum()}"

In [ ]:
# Define the maximum number of characters in an ad
max_size_characters = 1500

# How many ads have more than this maximum number of characters?
f"Number of ads with more than {max_size_characters} characters: {(df['ad_creative_bodies'].apply(len) > max_size_characters).sum()}"

In [ ]:
# We want to filter only the ads containing at least 20 characters and those below 1500 = majority of ads and we get rid of outliers

filter_char = (df['ad_creative_bodies'].apply(len) >= minimum_size_characters) & (df['ad_creative_bodies'].apply(len) <= max_size_characters)

# apply to entire dataframe
df_char = df[filter_char]
df_char.shape

#Full dataframe

In [ ]:
df = df_char

df

In [ ]:
df.info()

# Cleaning/Preprocessing of the text columns

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocessing(sentence):
    # Removing whitespaces
    sentence = sentence.strip()
    # Lowercasing
    sentence = sentence.lower()
    # Removing numbers
    sentence = ''.join(char for char in sentence if not char.isdigit())
    # Removing punctuation
    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, '')
    # Tokenizing
    tokenized = word_tokenize(sentence)
    # Lemmatizing
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in tokenized]
    cleaned_sentence = " ".join(lemmatized)
    return cleaned_sentence

In [ ]:
df_copy = df.copy()

#Clean entire df

In [ ]:
df_copy['translated_posts'] = df_copy['translated_posts'].apply(preprocessing)

# **Sentiment Analysis using the BERT model via Hugging Face**

**Step 1 - sample analysis**

In [ ]:
# create a function that samples the dataset called 'sample' and takes in 2 parameters: the dataframe, and the value of the fraction we want. the value is a float.
def sample(df, value):
    df_sample = df.sample(frac=value)
    return df_sample

In [ ]:
!pip install requests

In [ ]:
import requests
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, pipeline

#Model

In [ ]:
classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=10)

In [ ]:
df_sample = sample(df_copy, 0.02)

In [ ]:
# Extract the translated text from the 'ad_creative_bodies_translated' column for one row
translated_texts = list(df_sample['translated_posts'].dropna())

In [ ]:
# Perform sentiment analysis on the translated content for one row
sentiment_results = classifier(translated_texts)

In [ ]:
df_model = df_sample.copy()

In [ ]:
# Create a new column 'Top_10_Sentiments' with the top 10 sentiments for each row
df_model['top_10_sentiments'] = df_model['translated_posts'].apply(
    lambda text: [{'label': label['label'], 'score': label['score']} for label in classifier([text])[0]]
)

# Loaded model results in google drive

In [ ]:
df_model.to_csv('drive/My Drive/project_trained.csv')

# **Method 2 - function literal_eval()**

In [ ]:
import json
import ast

In [ ]:
df = df_model

In [ ]:
sentiments = [ast.literal_eval(x) for x in df.top_10_sentiments]
all_labels = []
for s in sentiments:
  for x in s:
    all_labels.append(x["label"])
all_labels = list(set(all_labels))

df2 = []
for s in sentiments:
  row = {label: 0 for label in all_labels}
  for x in s:
    row[x["label"]] = x["score"]
  df2.append(row)
df2 = pd.DataFrame(df2)
df2["party"] = df["parties"]
df2

In [ ]:
df3 = {"party": [], "emotion": [], "score": []}

for label in all_labels:
  df3["party"].extend(df2["party"])
  df3["emotion"].extend([label]*len(df2))
  df3["score"].extend(df2[label])

df3 = pd.DataFrame(df3)
df3

In [ ]:
df3['emotion'].unique()

In [ ]:
import math

# Define color mapping for each political party
party_colors = {
    'CDU_CSU': 'black',
    'FDP': 'yellow',
    'SPD': 'red',
    'AfD': 'lightblue',
    'LINKE': 'pink',
    'GRÜNE': 'green'
}

# df3 is the DataFrame with multiple sentiments
sentiments = df3['emotion'].unique()

# number of rows and columns for the grid
num_plots = len(sentiments)
num_cols = 3
num_rows = math.ceil(num_plots / num_cols)

# grid of subplots for each sentiment
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 5 * num_rows))

# Flatten the axes array if it's a multi-dimensional array
axes = axes.flatten()

# Loop through each sentiment to create 27 bar plots
for i, sentiment in enumerate(sentiments):
    subset_df = df3.query(f"emotion=='{sentiment}'")

    # Check if all parties have data for the current sentiment
    for party in party_colors.keys():
        if party not in subset_df['party'].unique():
            party_colors[party] = 'gray' # If a party is missing, default color is 'gray'
    sns.barplot(data=subset_df, x="party", y="score", ax=axes[i], palette=party_colors)
    axes[i].set_title(f"Prevalence of {sentiment}", fontsize=10, weight='bold', color='navy')

# prevent overlapping titles
plt.tight_layout()

# plots
plt.show

sns.set(rc={"figure.dpi":300, 'savefig.dpi':300})
sns.set_context('notebook')
sns.set_style("ticks")

#plot = sns.barplot(data=df3.query("emotion=='disgust'"), x="party", y="score")
#plot.set_title('Prevalence of Disgust', fontsize=25, weight='bold', color='navy')
plot.set_xlabel('Party', fontsize=8)
plot.tick_params(labelsize=8)

fig = plot.get_figure()
fig.savefig('Chart_all.png')

In [ ]:
nrows=4
ncols=1
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5,8))

for i in range(nrows*ncols):
  row = i%nrows
  ax = axs[i]
  label = all_labels[i]
  sns.barplot(data=df3.query("emotion==@label"), x="party", y="score",ax=ax)
  ax.set_title(label)

fig.tight_layout()

**We have decided to select 'love', 'caring','disgust', 'fear' in our final presentation - let"s select and also amplify the titles.**

In [ ]:
sns.set(rc={"figure.dpi":300, 'savefig.dpi':300})
sns.set_context('notebook')
sns.set_style("ticks")

plot = sns.barplot(data=df3.query("emotion=='love'"), x="party", y="score")
plot.set_title('Prevalence of Love', fontsize=25, weight='bold', color='navy')
plot.set_xlabel('Party', fontsize=12)
plot.tick_params(labelsize=12)

fig = plot.get_figure()
fig.savefig('Chart_love.png')

# **Plotting**